PROTOTYPE
# **Medical Diagnostic Assistant**: AI-Powered RAG

---

## **Value Proposition**

In order to reduce the risk of AI-generated medical hallucinations, a RAG system grounded in **The Merck Manual** was developed to constrain clinical responses to verified source material, achieving average **groundedness of 4.8/5** across 9 validated clinical queries in the current run.

---

## **Executive Summary**

### Business Opportunities

* **Opportunity A:** AI hallucinations in healthcare create unacceptable clinical risk. Unconstrained LLMs produce fluent but unverifiable medical guidance that cannot be safely deployed.

* **Solution A:** The RAG architecture constrains generated responses to content retrieved from **The Merck Manual**, making answers evidence-linked and easier to review.

* **Opportunity B:** Healthcare professionals face increasing information overload and need faster access to trusted medical reference material.

* **Solution B:** The system retrieves relevant passages and generates structured responses in seconds, reducing manual search effort and improving answer consistency.

* **Opportunity C:** Multiple integration paths exist across healthcare and clinical decision-support markets.

* **Solution C** (Options): Internal clinical decision-support dashboard for diagnostic teams. API integration into electronic health record (EHR) systems. Mobile reference tool for emergency and field medicine providers. In underdeveloped countries, the same architecture can help focus limited clinical resources by quickly surfacing high-value guidance on top of existing care capacity.

### Outcomes

* **Model Performance**

  * In this run, the RAG system achieved strong overall quality across 9 clinical queries, with average **groundedness = 4.8/5** and **relevance = 3.9/5**.
  * Groundedness was the standout result: **7 of 9** responses scored a perfect **5/5**, showing that the system consistently stayed anchored to the retrieved medical evidence.
  * Compared with the ungrounded baseline, the RAG workflow delivered materially more reliable and reviewable answers by reducing unsupported specificity and adding source traceability.

* **Architecture**

  * Retrieval-Augmented Generation with **`k=3`** and **`temperature=0.1`** produced the best final answer quality.
  * The medical reference corpus loaded **4,114 pages** and was indexed into **18,020 chunks** in **Chroma** for retrieval.

* **Economic Impact**

  * Reducing clinical information retrieval time and improving answer traceability can support both workflow efficiency and lower operational risk.

* **Strategy Recommendation**

  * **Enterprise:** Deploy as a clinical decision-support tool integrated into existing EHR workflows, with audit logging for compliance.
  * **SMB:** Offer as a standalone reference application for smaller practices and field clinicians requiring rapid, grounded medical guidance.

---

## **Problem Space**

### Overview

* Healthcare professionals face information overload, struggling to synthesize vast medical literature under time pressure.
* Unconstrained AI models risk producing fluent but unsupported medical guidance — a critical liability in clinical settings.
* Grounding AI responses in authoritative sources ensures traceability and clinical reliability.

### Data Description

* Source document: **The Merck Manual of Diagnosis & Therapy**
* Data file: `medical_diagnosis_manual.pdf`
* Current corpus size: **4,114 pages**
* Current retrieval index size: **18,020 chunks**

### Clinical Queries Evaluated

1. Diagnostic Assistance: Pulmonary embolism symptoms and treatments
2. Surgical Inquiry: Appendicitis symptoms, medicine vs. surgical treatment
3. Drug Information: Trade names of hypertension medications
4. Treatment Plans: Rheumatoid arthritis first-line options and alternatives
5. Dermatology: Sudden patchy hair loss causes and treatments
6. Neurology: Brain injury treatment recommendations
7. Emergency Care: Leg fracture precautions and treatment steps
8. Specialty Knowledge: Diagnostic steps for endocrine disorders
9. Critical Care Protocols: Sepsis management in critical care

### Process

* A baseline ungrounded LLM was evaluated first.
* Prompt and decoding settings were tested for tone and consistency.
* The Merck Manual was chunked, embedded, and indexed in **Chroma** for semantic retrieval.
* RAG generation was tuned across multiple `k` and temperature settings, with the final configuration set to **`k=3`, `temp=0.1`**.
* Automated evaluation scored outputs on groundedness and relevance using a 1–5 scale.

### Results
>| Model | Groundedness | Relevance | Traceability |
>|---|---|---|---|
>| Model 1: Baseline LLM (Ungrounded) | Low | Medium | None |
>| **Model 2: RAG LLM (`k=3`, `temp=0.1`) ✅** | **4.8/5** | **3.9/5** | **Page-cited** |

# **Execute Code**
*   Installing Colab Dependecies may require a Runtime restart. You will be notified if a restart is required
*   Data cited in markdown cells may differ sligntly from outputs, as training data and test data are randomized

## Environment Initialization

### Environment Setup and Dependency Configuration

**Process:** The runtime checks whether the environment is properly provisioned for local inference, PDF loading, embeddings, and vector search. If dependencies are missing, they are installed and a runtime restart is prompted; if the environment is already provisioned, execution continues without interruption.

**Outcome:** Execution continued without a restart: the required dependencies were already installed.

In [ ]:
import os
from pathlib import Path
from IPython.display import display, HTML

# Set flags for llama-cpp-python to compile with CUDA support for faster local inference
os.environ["CMAKE_ARGS"] = "-DLLAMA_CUBLAS=on"
os.environ["FORCE_CMAKE"] = "1"

flag_file = Path("/content/.deps_installed_flag")

if not flag_file.exists():
    print("Installing dependencies... This may take a minute or two.")

    # Safely install dependencies, suppressing output and error warnings
    !pip install -q llama-cpp-python==0.2.28 > /dev/null 2>&1
    !pip install -q huggingface_hub pandas tiktoken pymupdf langchain langchain-community langchain-text-splitters chromadb sentence-transformers transformers > /dev/null 2>&1

    # Set the persistent flag so this block skips after restarting
    flag_file.touch()

    display(HTML("""
    <div style="
        border: 4px solid #d93025;
        background: #fce8e6;
        color: #202124;
        padding: 24px;
        border-radius: 12px;
        font-family: Arial, sans-serif;
        margin: 16px 0;
    ">
        <div style="font-size: 30px; font-weight: 800; color: #b31412; margin-bottom: 12px;">
            ⚠️ Runtime Restart Required
        </div>
        <div style="font-size: 22px; line-height: 1.5; font-weight: 600;">
            Dependencies have been successfully installed!
            <br><br>
            Please go to <b>Runtime → Restart session and run all</b> in the top menu.
        </div>
    </div>
    """))
else:
    print("✅ Dependencies already installed. No restart required.")


✅ Dependencies already installed. No restart required.


### Core Imports and Shared Configuration

**Process:** Shared libraries, retrieval components, and the 9-question evaluation set are loaded once for the full workflow.

**Outcome:** Baseline inference, prompt tuning, RAG generation, and evaluation all run on the same model stack and benchmark set.

In [ ]:
# ============================================================
# Core Imports and Shared Configuration
# ============================================================

import os  # for filesystem checks and local path handling
import pandas as pd  # for tabular result handling and evaluation outputs
from huggingface_hub import hf_hub_download  # for downloading the quantized GGUF model from Hugging Face
from llama_cpp import Llama  # for local LLM inference using llama-cpp
from langchain_text_splitters import RecursiveCharacterTextSplitter  # for chunking long medical text into retrieval units
from langchain_community.document_loaders import PyMuPDFLoader  # for loading the medical reference PDF into documents
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings  # for converting chunks into embeddings
from langchain_community.vectorstores import Chroma  # for storing and retrieving embedded chunks

# Global Qs
questions = [
    "What are the common symptoms and treatments for pulmonary embolism?",
    "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
    "Can you provide the trade names of medications used for treating hypertension?",
    "What are the first-line options and alternatives for managing rheumatoid arthritis?",
    "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
    "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",
    "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?",
    "What are the diagnostic steps for suspected endocrine disorders?",
    "What is the protocol for managing sepsis in a critical care unit?"
]

## Baseline Medical Question Answering (Ungrounded)

### Load and Initialize LLM

**Process:** A quantized **Mistral-7B Instruct** GGUF model is downloaded from Hugging Face and loaded into `llama-cpp` with GPU offload.

**Analysis:** The model is public and loads successfully, but initialization still depends on a large model download and compatible runtime settings.

**Outcome:** A local ungrounded baseline model is initialized for comparison.

In [ ]:
# ============================================================
# Load and Initialize LLM
# ============================================================


# Load LLM
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q4_K_M.gguf"

model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

# Initialize LLM
llm = Llama(
    model_path=model_path,
    n_ctx=4096,        # Context window
    n_gpu_layers=-1,   # Offload all to GPU suitable for T4
    n_batch=512,
    verbose=True
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


mistral-7b-instruct-v0.2.Q4_K_M.gguf:   0%|          | 0.00/4.37G [00:00<?, ?B/s]

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


### Inference Wrapper

**Process:** Prompt construction and decoding controls are standardized in a reusable wrapper for all baseline calls.

**Outcome:** Baseline responses are generated through a consistent inference path.

In [ ]:
# ============================================================
# Inference Wrapper
# ============================================================

def generate_response(query, max_tokens=2048, temperature=0.1, top_p=0.95, top_k=50, system_prompt=None):
    if system_prompt:
        prompt = f"[INST] {system_prompt}\n{query} [/INST]"
    else:
        prompt = f"[INST] {query} [/INST]"

    model_output = llm(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k
    )
    return model_output['choices'][0]['text'].strip()

### Baseline Question Execution

**Process:** All 9 medical benchmark questions are run through the ungrounded model and stored for later comparison.

**Analysis:** The responses are fluent, but the saved outputs show unsupported specificity, especially on the hypertension trade-name question.

**Outcome:** A full baseline set is established, along with a clear hallucination pattern to improve on.

In [ ]:
# ============================================================
# Baseline Question Execution
# ============================================================

vanilla_results = []
for i, q in enumerate(questions, 1):
    print(f"--- Question {i} ---")
    print(f"Q: {q}")
    ans = generate_response(q)
    print(f"A: {ans}\n")
    vanilla_results.append(ans)

--- Question 1 ---
Q: What are the common symptoms and treatments for pulmonary embolism?
A: A pulmonary embolism (PE) is a blockage in one or more arteries of the lungs, usually caused by a blood clot that travels from elsewhere in the body. Here are some common symptoms of PE:

1. Shortness of breath: This is the most common symptom and can range from mild to severe. The person may feel like they can't get enough air or that they're breathing too hard.
2. Chest pain: PE can cause sharp, stabbing chest pain that worsens with deep breaths or physical activity. The pain may be worse when lying down.
3. Rapid heart rate: PE can cause the heart to beat faster as it tries to pump blood through the blocked artery.
4. Coughing up blood: In some cases, PE can cause coughing up blood or bloody mucus.
5. Swelling: PE can cause swelling in the legs, ankles, and feet due to fluid buildup.
6. Rapid breathing: PE can cause rapid, shallow breathing as the body tries to get more oxygen.
7. Anxiety: P

Llama.generate: prefix-match hit


A: Appendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right side of the abdomen. The following are common symptoms of appendicitis:

1. Abdominal pain: This is usually the first and most prominent symptom. The pain may start as a mild discomfort in the lower right abdomen, but it can quickly develop into a sharp, persistent pain.
2. Loss of appetite: People with appendicitis often lose their appetite due to abdominal pain and nausea.
3. Nausea and vomiting: Vomiting is a common symptom of appendicitis, especially as the condition progresses.
4. Fever: A fever may develop as the body responds to the infection.
5. Constipation or diarrhea: Some people with appendicitis experience constipation, while others have diarrhea.
6. Abdominal swelling: The abdomen may become swollen and tender to the touch.
7. Inability to pass gas: This can be a sign of obstruction of the appendix.

Appendicitis cannot be cured with m

Llama.generate: prefix-match hit


A: Yes, I can provide a list of some common trade names of medications used for treating hypertension (high blood pressure). Please note that this is not an exhaustive list and there are many other brand names available depending on the specific type and formulation of the medication.

1. Diuretics:
   - Hydrochlorothiazide (Microzide, Esidrix)
   - Furosemide (Lasix)
   - Spironolactone (Aldactone)
   - Chlorthalidone (Hygroton, Thalitone)

2. Beta-blockers:
   - Metoprolol (Lopressor, Toprol XL)
   - Propranolol (Inderal LA, Inderal XL)
   - Atenolol (Tenormin)
   - Bisoprolol (Zebeta)

3. Calcium channel blockers:
   - Nifedipine (Procardia, Adalat CC)
   - Amlodipine (Norvasc)
   - Diltiazem (Cardizem, Tiazac)
   - Verapamil (Calan SR, Isoptin SR)

4. Angiotensin-converting enzyme (ACE) inhibitors:
   - Lisinopril (Prinivil, Zestril)
   - Enalapril (Vasotec)
   - Quinapril (Accupril)
   - Benazepril (Lotensin)

5. Angiotensin II receptor blockers (ARBs):
   - Losartan (Cozaar)
   -

...

A: If a person has fractured their leg during a hiking trip, it is important that they receive proper medical attention as soon as possible. Here are some necessary precautions and treatment steps:

1. Assess the situation: Check the severity of the injury and assess the overall condition of the person. If the fracture is open or the person is in severe pain, distress, or unable to move, do not attempt to move them without professional help. Call for emergency medical assistance immediately.

2. Provide first aid: If the person's condition permits, provide first aid by immobilizing the leg with a splint or sling. Use any available materials such as sticks, branches, or clothing to create a makeshift splint. Be careful not to move the leg excessively and avoid putting weight on it.

3. Keep the person warm and comfortable: Provide shelter from the elements, such as rain or extreme temperatures, and keep the person hydrated and nourished. If possible, establish a signal for rescue teams 

Llama.generate: prefix-match hit


A: 1. Thorough Clinical Evaluation: The first step in diagnosing suspected endocrine disorders is a comprehensive clinical evaluation. This includes taking a detailed medical history, performing a physical examination, and assessing symptoms such as weight changes, fatigue, and skin changes.

2. Laboratory Tests: A variety of laboratory tests may be ordered to help diagnose endocrine disorders. These tests can include:
   - Blood glucose level: To check for diabetes or other metabolic disorders.
   - Hormone levels: To measure the levels of various hormones in the body, such as insulin, thyroid hormones, growth hormone, cortisol, and sex hormones.
   - Imaging studies: To visualize glands and organs involved in endocrine function, such as the pituitary gland, thyroid gland, pancreas, and adrenal glands.
   - Genetic testing: In some cases, genetic testing may be recommended to identify inherited endocrine disorders.

3. Diagnostic Imaging: Imaging studies, such as ultrasound, CT scan, 

Llama.generate: prefix-match hit


A: Sepsis is a life-threatening condition caused by the body's response to an infection, and managing it effectively in a critical care unit requires a multidisciplinary approach. Here are some key steps in the protocol for managing sepsis in a critical care unit:

1. Early recognition and diagnosis: Identify sepsis early based on clinical suspicion and laboratory results, such as elevated white blood cell count, lactate level, and procalcitonin. Use tools like the Sequential Organ Failure Assessment (SOFA) score or Quick Sequential Organ Failure Assessment (qSOFA) to help identify patients at risk for sepsis.
2. Resuscitation: Begin resuscitation as soon as possible with intravenous fluids and oxygen, aiming for a mean arterial pressure of 65 mmHg or higher and adequate oxygen saturation. Consider administering vasopressors if blood pressure is not responding to fluid resuscitation.
3. Antibiotics: Administer broad-spectrum antibiotics as soon as possible based on suspected infection 

## Prompt Engineering and Decoding Control (Parameter Tuning)

### Prompt Configuration Evaluation

**Process:** Five prompt-and-decoding profiles are tested across the full question set to compare tone, structure, and determinism.

**Analysis:** The sweep improves presentation and consistency, but it does not add evidence; unsupported details still appear when the model answers from priors alone.

**Outcome:** Prompt engineering improves format, not grounding.

In [ ]:
# ============================================================
# Prompt Configuration Evaluation
# ============================================================

# Define 5 Configurations

configs = [
    {"name": "High Creativity", "temp": 0.9, "sys": "You are a helpful medical assistant.", "top_k": 100},
    {"name": "Strict Professional", "temp": 0.0, "sys": "You are a concise, professional doctor. Answer only what is asked.", "top_k": 10},
    {"name": "Chain of Thought", "temp": 0.3, "sys": "Think step-by-step. Explain the reasoning before giving the final answer.", "top_k": 40},
    {"name": "ELI5", "temp": 0.7, "sys": "Explain like I am 5 years old.", "top_k": 50},
    {"name": "Balanced", "temp": 0.4, "sys": "You are a knowledgeable assistant. Provide detailed clinical information.", "top_k": 50}
]

# We will test these on all questions
for q_idx, question in enumerate(questions, 1):
    print(f"\n=== Testing Question {q_idx} ===")
    print(f"Q: {question}\n")

    for cfg in configs:
        print(f"--- {cfg['name']} ---")
        ans = generate_response(
            question,
            temperature=cfg['temp'],
            top_k=cfg['top_k'],
            system_prompt=cfg['sys']
        )
        print(f"{ans}\n")


=== Testing Question 1 ===
Q: What are the common symptoms and treatments for pulmonary embolism?

--- High Creativity ---


Llama.generate: prefix-match hit


A pulmonary embolism (PE) is a serious condition caused by a blockage in one or more arteries of the lungs, usually due to a blood clot that travels from elsewhere in the body. Here are some common symptoms and treatments for PE:

Symptoms:
1. Shortness of breath: This is often the most prominent symptom of a PE. You may feel like you can't catch your breath or that you're winded, even at rest.
2. Chest pain: PE can cause sharp or dull chest pain, especially when you take deep breaths or cough.
3. Rapid heart rate: Your heart may beat faster than normal in response to the blockage.
4. Coughing up blood: In some cases, PE can cause you to cough up blood, which can be a sign of a more severe condition.
5. Swelling: You may experience swelling in your legs, ankles, or arms due to blood clots forming in the deep veins of your legs (deep vein thrombosis) and traveling to your lungs.
6. Fatigue: PE can make you feel tired or weak.
7. Anxiety: Some people with PE may feel anxious or experienc

Llama.generate: prefix-match hit


A pulmonary embolism (PE) is a blockage in the arteries of the lungs, usually caused by a blood clot that travels from elsewhere in the body. Common symptoms include:

1. Shortness of breath
2. Chest pain or discomfort, often worsening with deep breathing or coughing
3. Rapid heart rate
4. Anxiety
5. Coughing up blood
6. Swelling in the legs or arms
7. Fainting

Treatment for PE depends on the severity of the blockage and the overall health of the patient. Options include:

1. Anticoagulant medications to prevent clot formation and growth
2. Thrombolytic therapy, which involves administering drugs that dissolve blood clots
3. Embolectomy, a surgical procedure to remove the clot from the lung artery
4. Supportive measures such as oxygen therapy, fluid replacement, and pain management.

It's important to note that PE can be life-threatening if not treated promptly, so seeking medical attention right away is crucial.

--- Chain of Thought ---


...

Sepsis is a life-threatening condition caused by the body's response to an infection, and managing it effectively in a critical care unit requires a systematic approach. Here are the key steps in the protocol for managing sepsis in a critical care unit:

1. Early recognition: The first step is to recognize sepsis as early as possible. This includes identifying patients who are at high risk of developing sepsis, such as those with infections, chronic illnesses, or compromised immune systems. Clinical suspicion should be based on the presence of infection and systemic inflammatory response syndrome (SIRS) criteria: body temperature above 38°C or below 36°C, heart rate greater than 90 beats per minute, respiratory rate greater than 20 breaths per minute, or mean arterial pressure less than 70 mmHg.

2. Rapid resuscitation: Once sepsis is suspected, rapid resuscitation is essential to prevent progression to septic shock. This includes administering intravenous fluids to maintain adequate b

Llama.generate: prefix-match hit


Sepsis is when your body gets really sick from an infection, and we need to help it get better in the hospital's special room called the Critical Care Unit. Here are some steps the doctors and nurses follow to make you feel better:

1. Give you lots of fluids through a big needle in your hand or arm to keep you hydrated.
2. Give you strong medicines to help fight the infection and lower your fever.
3. Monitor your heart rate, breathing, blood pressure, and oxygen levels very closely.
4. Check your blood tests frequently to see how well your organs are working.
5. Provide you with nutrients through a tube in your stomach if you can't eat normally.
6. Keep you comfortable and quiet so your body can focus on healing.
7. Work closely with other doctors and specialists as needed.
8. Encourage you to move around or perform deep breathing exercises when you feel better.

The team in the Critical Care Unit works together to ensure that you receive appropriate care based on your individual need

Llama.generate: prefix-match hit


Sepsis is a life-threatening condition caused by the body's response to an infection, which can lead to tissue damage and organ failure. Management of sepsis in a critical care unit involves a multidisciplinary approach that includes early recognition, rapid resuscitation, source control, and supportive care. Here are the key steps in managing sepsis in a critical care unit:

1. Early Recognition: Identify sepsis suspects based on clinical suspicion, laboratory findings, or the use of screening tools such as the Sequential Organ Failure Assessment (SOFA) score or Quick Sequential Organ Failure Assessment (qSOFA) score.
2. Rapid Resuscitation: Initiate fluid resuscitation with isotonic crystalloids to maintain adequate tissue perfusion and organ function. Monitor hemodynamic parameters such as mean arterial pressure, central venous pressure, and urine output. Consider the use of vasopressors and/or inotropes if necessary to maintain adequate blood pressure and organ perfusion.
3. Source

## Reference Preparation for Retrieval-Augmented Generation

### Reference Document Ingestion

**Process:** The notebook downloads the managed dataset package, resolves the Merck Manual PDF, and loads it into page-level documents for retrieval preparation.

**Analysis:** This run loads **4,114 pages**, and the sampled opening pages are front matter, so retrieval quality—not page order—will determine answer quality.

**Outcome:** A single authoritative medical reference corpus is ready for chunking and indexing.

In [ ]:
# ============================================================
# Reference Document Ingestion
# ============================================================

import importlib.util  # for loading the shared data access helper dynamically
import shutil  # for clearing stale notebook-specific cached dataset folders
import sys  # for detecting whether the notebook is running in Colab
import urllib.request  # for downloading the shared helper file from GitHub
from pathlib import Path  # for cross-platform file path handling
from langchain_community.document_loaders import PyMuPDFLoader  # for parsing the PDF into page-level documents

HELPER_URL = "https://raw.githubusercontent.com/EvagAIML/000-website-notebooks/main/notebooks/_shared/data_access.py"

SLUG = "medical-diagnosis-assist__qa__rag-exp"
EXTRACT_TO = "medical-diagnosis-assist__qa__rag-exp__dataset__v1"
ZIP_NAME = "medical-diagnosis-assist__qa__rag-exp__dataset__v1.zip"

helper_path = Path("/content/data_access.py") if "google.colab" in sys.modules else Path("./data_access.py")
cache_root = Path("/content/data_cache") if "google.colab" in sys.modules else Path("./data_cache")
session_flag = Path("/content/.medical_cache_refreshed") if "google.colab" in sys.modules else Path(".medical_cache_refreshed")

# Clear stale notebook-specific artifacts once per fresh session
if not session_flag.exists():
    helper_path.unlink(missing_ok=True)
    shutil.rmtree(cache_root / EXTRACT_TO, ignore_errors=True)
    (cache_root / ZIP_NAME).unlink(missing_ok=True)
    session_flag.touch()

urllib.request.urlretrieve(HELPER_URL, helper_path)

spec = importlib.util.spec_from_file_location("data_access", helper_path)
data_access = importlib.util.module_from_spec(spec)
spec.loader.exec_module(data_access)

ensure_dataset = data_access.ensure_dataset

DATA_DIR = ensure_dataset(SLUG)
RAW_DIR = DATA_DIR / "raw"
doc_path = RAW_DIR / "medical_diagnosis_manual.pdf"

loader = PyMuPDFLoader(str(doc_path))
docs = loader.load()
print(f"Loaded {len(docs)} pages from {doc_path.name}.")

# Check first 5 pages
for i in range(min(5, len(docs))):
    print(f"--- Page {i+1} ---")
    print(docs[i].page_content[:500])
    print("...")

Extracting dataset for 'medical-diagnosis-assist__qa__rag-exp'...
Loaded 4114 pages from medical_diagnosis_manual.pdf.
--- Page 1 ---
erikvdesigner@gmail.com
U36PAIRLB4
ant for personal use by erikvdesigner@g
shing the contents in part or full is liable
...
--- Page 2 ---
erikvdesigner@gmail.com
U36PAIRLB4
This file is meant for personal use by erikvdesigner@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
...
--- Page 3 ---
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .................................
...
--- Page 4 ---
491
C

### Reference Text Chunking

**Process:** The manual is split into overlapping 1,000-character chunks with 200-character overlap to preserve local context during retrieval.

**Outcome:** The source corpus is converted into **18,020** retrieval units for semantic search.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(docs)
print(f"Total Chunks created: {len(chunks)}")

Total Chunks created: 18020


### Embedding & Vector Store

**Process:** All chunks are embedded with `all-MiniLM-L6-v2` and stored in Chroma, with the default retriever configured at `k=3`.

**Analysis:** This step turns the manual into a searchable evidence base, making retrieval quality the main driver of answer quality.

**Outcome:** A working semantic index is available for grounded medical retrieval.

In [ ]:
# ============================================================
# Embedding & Vector Store
# ============================================================

from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings

embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db_medical_final"
)

# Default Retriever
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

/tmp/ipykernel_18102/4154461508.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Retrieval-Augmented Medical Question Answering

### RAG Generation Function

**Process:** For each query, relevant Merck passages are retrieved, injected into the prompt, and used to constrain answer generation.

**Outcome:** Answers are now conditioned on retrieved evidence instead of model memory alone.

In [ ]:
# ============================================================
# RAG Generation Function
# ============================================================

def generate_rag_answer(query, k=3, temperature=0.1, max_tokens=2048):
    # Retrieve
    retrieved_docs = vector_db.similarity_search(query, k=k)

    # Format context and extract sources
    context_list = []
    sources_info = []
    for d in retrieved_docs:
        page_num = d.metadata.get('page', 'Unknown')
        # Assuming page is 0-indexed integer from PyMuPDF
        if isinstance(page_num, int):
            page_num += 1
        src_str = f"Page {page_num}"
        sources_info.append(src_str)
        context_list.append(f"[{src_str}] {d.page_content}")

    context = "\n\n".join(context_list)
    unique_sources = ", ".join(sorted(list(set(sources_info))))

    # Construct Prompt
    sys_msg = "You are a professional medical assistant for healthcare professionals. Use the following Context to answer the Question. If the answer is not in the context, say so. Do not advise consulting a healthcare professional."
    user_msg = f"Context:\n{context}\n\nQuestion: {query}\n\nPlease cite specific Page numbers and Sections from the text in your answer."

    prompt = f"[INST] {sys_msg}\n{user_msg} [/INST]"

    # Generate
    output = llm(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        stop=["[/INST]"]
    )
    return output['choices'][0]['text'].strip(), context

### RAG Parameter Tuning (5 Combinations)

**Process:** Five retrieval and generation settings vary context depth and temperature across the full question set to determine the best balance of evidence coverage, specificity, and hallucination control.

**Analysis:** Lower-temperature settings produced more disciplined answers that stayed closer to the retrieved text. Increasing `k` improved coverage when evidence was spread across multiple passages, but pushing context too high introduced noise, while higher-temperature settings made responses more verbose and less tightly grounded.

**Outcome:** The final configuration selected was **`k=3`, `temperature=0.1`** because it provided the best balance of sufficient context, stable answer quality, and strong evidence adherence without unnecessary noise.

In [ ]:
# ============================================================
# RAG Parameter Tuning (5 Combinations)
# ============================================================

rag_configs = [
    {"name": "C1: Low Context, Precise", "k": 2, "temp": 0.0},
    {"name": "C2: Standard", "k": 3, "temp": 0.1},
    {"name": "C3: High Context", "k": 5, "temp": 0.1},
    {"name": "C4: Creative", "k": 3, "temp": 0.7},
    {"name": "C5: Max Context, Strict", "k": 7, "temp": 0.0}
]

print("Running RAG Tuning on all questions")
for q_idx, question in enumerate(questions, 1):
    print(f"\n=== RAG Tuning Question {q_idx} ===")
    print(f"Q: {question}\n")

    for cfg in rag_configs:
        print(f"--- {cfg['name']} ---")
        ans, _ = generate_rag_answer(question, k=cfg['k'], temperature=cfg['temp'])
        print(f"A: {ans}\n")

Running RAG Tuning on all questions

=== RAG Tuning Question 1 ===
Q: What are the common symptoms and treatments for pulmonary embolism?

--- C1: Low Context, Precise ---


Llama.generate: prefix-match hit


A: According to the context on pages 2080 and 2081 of Chapter 194, the common symptoms for pulmonary embolism include dyspnea (shortness of breath), pleuritic chest pain, cough, syncope or cardiorespiratory arrest in severe cases, tachypnea (rapid breathing), tachycardia (fast heart rate), hypotension (low blood pressure), and a loud pulmonic component of the 2nd heart sound.

The treatment for pulmonary embolism includes anticoagulants to prevent clot formation and growth, and in some cases, clot dissolution with thrombolytics or surgical removal. Preventive measures include anticoagulants and the insertion of an inferior vena caval filter (Section: "Treatment" on page 2080).

Therefore, the answer is: The common symptoms for pulmonary embolism are dyspnea, pleuritic chest pain, cough, syncope or cardiorespiratory arrest in severe cases, tachypnea, tachycardia, hypotension, and a loud pulmonic component of the 2nd heart sound (Page 2080). The treatment includes anticoagulants, thrombo

Llama.generate: prefix-match hit


A: According to the context on pages 2080 and 2081, the common symptoms of pulmonary embolism include dyspnea, pleuritic chest pain, cough, tachypnea, tachycardia, hypotension, and a loud pulmonic component of the 2nd heart sound. (Section: Introduction)

The treatment for pulmonary embolism involves anticoagulants, clot dissolution with thrombolytics, or surgical removal of the clots. Preventive measures include anticoagulants and inferior vena caval filters. (Section: Treatment)

Please note that this information is based on the provided context and may not cover all possible symptoms or treatments for pulmonary embolism. For comprehensive medical advice, consult a healthcare professional. (Do not advise consulting a healthcare professional)

--- C3: High Context ---


...

A: According to the context provided, the protocol for managing sepsis in a critical care unit involves several steps. First, supportive care includes keeping the patient warm, controlling hemorrhage, checking the airway and ventilation, and providing supplemental oxygen (Page 2401). If the patient is experiencing septic shock, treatment begins with airway intubation and mechanical ventilation, large IV catheters, and potentially a central venous line or intraosseous needle for fluid resuscitation (Page 2454).

The specific management of sepsis includes providing oxygen, airway intubation and mechanical ventilation if necessary, inserting two large IV catheters, and considering a central venous line or intraosseous needle for fluid resuscitation. Antibiotics are given empirically after appropriate cultures are obtained (Page 2454). The treatment also includes surgical excision of infected or necrotic tissues and drainage of pus, supportive care, and sometimes intensive control of blood

Llama.generate: prefix-match hit


A: According to the context provided on pages 2453-2454 of your text, the protocol for managing sepsis in a critical care unit includes the following steps:

1. Provide first aid: Keep the patient warm, control hemorrhage if necessary, secure the airway and ventilation, and turn the patient's head to one side to prevent aspiration if emesis occurs. Nothing should be given by mouth. (Chapter 222, Introduction)
2. Provide supplemental oxygen: Use a face mask to provide oxygen to the patient. (Chapter 222, Patient Monitoring and Testing)
3. Establish intravenous access: Insert two large IV catheters into separate peripheral veins or use a central venous line or intraosseous needle if peripheral veins cannot be accessed promptly. (Chapter 222, Supportive care for the ICU patient)
4. Initiate fluid resuscitation: Provide aggressive fluid resuscitation to address critical reduction in tissue perfusion. (Chapter 227, Introduction)
5. Administer antibiotics: Begin antibiotic therapy as soon as

Llama.generate: prefix-match hit


A: According to the context provided in "The Merck Manual of Diagnosis & Therapy, 19th Edition," the protocol for managing sepsis in a critical care unit involves several steps. (Chapter 227. Sepsis & Septic Shock, pages 2453-2455)

1. Patient Monitoring and Supportive Care: The patient's vital signs, including temperature, heart rate, respiratory rate, blood pressure, and oxygen saturation, should be closely monitored. Warmth is provided to keep the patient comfortable. Hemorrhage is controlled, and airway and ventilation are checked. Nothing is given by mouth, and the patient's head is turned to one side if emesis occurs to avoid aspiration. (Chapter 222. Approach to the Critically Ill Patient, page 2401)

2. Initial Assessment and Treatment: Septic shock is a medical emergency that requires immediate attention. The patient's airway, breathing, circulation, and disability (availability of consciousness) are assessed and addressed accordingly. Supplemental oxygen is provided, and if t

### Final RAG Configuration -> Answers (Configuration: k=3, temp=0.1)

**Process:** The selected RAG configuration is run across all 9 questions, and both generated answers and retrieved contexts are stored.

**Analysis:** The final answers are tighter than baseline: they cite pages, stay closer to the source text, and avoid the unsupported trade-name behavior seen in the ungrounded run.

**Outcome:** A complete grounded answer set is produced for scoring and review.

In [ ]:
# ============================================================
# Final RAG Configuration -> Answers (Configuration: k=3, temp=0.1)
# ============================================================

rag_responses = []
rag_contexts = []

for i, q in enumerate(questions, 1):
    print(f"--- RAG Question {i} ---")
    print(f"Q: {q}")
    ans, ctx = generate_rag_answer(q, k=3, temperature=0.1)
    print(f"A: {ans}\n")
    rag_responses.append(ans)
    rag_contexts.append(ctx)

--- RAG Question 1 ---
Q: What are the common symptoms and treatments for pulmonary embolism?


Llama.generate: prefix-match hit


A: According to the context on pages 2080 and 2081, the common symptoms of pulmonary embolism include dyspnea, pleuritic chest pain, cough, tachypnea, tachycardia, hypotension, and a loud pulmonic component of the 2nd heart sound. (Section: Introduction)

The treatment for pulmonary embolism involves anticoagulants, clot dissolution with thrombolytics, or surgical removal of the clots. Preventive measures include anticoagulants and inferior vena caval filters. (Section: Treatment)

Please note that this information is based on the provided context and may not cover all possible symptoms or treatments for pulmonary embolism. For comprehensive medical advice, it's always best to consult a healthcare professional. (Sections: Introduction, Treatment)

--- RAG Question 2 ---
Q: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?


Llama.generate: prefix-match hit


A: The common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia, which is then followed by a shift in pain to the right lower quadrant. Pain increases with cough and motion. Classic signs are direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine). Additional signs are pain felt in the right lower quadrant with palpation of the left lower quadrant (Merck Manual, p. 174, Symptoms and Signs section).

Appendicitis cannot be cured via medicine alone as it requires surgical removal to prevent complications such as necrosis, gangrene, perforation, abscess formation, or inflammatory masses (Merck Manual, p. 174, Etiology and Treatment sections). The surgical procedure for treating appendicitis is typically an appendectomy, which may involve removing the appendix alone if it is not yet perforated or draining an abscess if

...

A: Based on the information provided in The Merck Manual of Diagnosis & Therapy, 19th Edition, here are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip:

1. The person should be kept off their feet for several days to promote better healing (Merck Manual, Chapter 322, Page 3387). This can be achieved by using crutches.
2. Wound care is essential. The wound should be kept clean and dry, and nonadherent and impermeable dressings should be applied (Merck Manual, Chapter 322, Page 3387). Antibiotic ointment should be applied daily until the wound closure device is removed.
3. For life- or limb-threatening injuries, hemorrhagic shock should be treated in the emergency department (Merck Manual, Chapter 323, Page 3391). Injuries to arteries are repaired surgically unless they affect only small arteries with good collateral circulation.
4. Severed nerves are surgically repaired; for neuropraxia and axonotmesis, initial treatment is us

Llama.generate: prefix-match hit


A: Based on the context provided, the diagnostic steps for suspected endocrine disorders include:

1. Measuring levels of peripheral endocrine hormones and pituitary hormones in the blood (Page 895 and 947).
2. Detecting autoantibodies to each affected glandular tissue to differentiate Primary Disorders of Specific Endocrines (PDS) from other causes (Page 947).
3. Elevated levels of pituitary tropic hormones suggest the hypothalamic-pituitary axis is intact, although some patients with type II PDS may have hypothalamic-pituitary insufficiency (Page 947).
4. Lifelong follow-up for unrecognized hypoparathyroidism or adrenal insufficiency, which can be life-threatening (Page 947).
5. Relatives should be made aware of the diagnosis and screened when appropriate; measurement of glutamic acid decarboxylase antibodies may be useful in determining risk (Page 947).

Therefore, the diagnostic steps involve a combination of clinical evaluation, laboratory testing, and family screening to confirm 

Llama.generate: prefix-match hit


A: According to the context provided on pages 2453-2454 of "Critical Care Medicine," the protocol for managing sepsis in a critical care unit includes the following steps:

1. Provide first aid: Keep the patient warm, control hemorrhage, check and secure the airway, and provide respiratory assistance if necessary (p. 2453).
2. Begin treatment simultaneously with evaluation: Provide supplemental oxygen via face mask, intubate and mechanically ventilate if necessary, insert two large IV catheters into separate peripheral veins, or use a central venous line or intraosseous needle if peripheral veins cannot be promptly accessed (p. 2453).
3. Aggressive fluid resuscitation: Provide intravenous fluids to maintain adequate blood pressure and tissue perfusion (p. 2454).
4. Antibiotics: Administer antibiotics to treat the underlying bacterial infection (p. 2454).
5. Surgical intervention: Excise infected or necrotic tissues and drain pus as necessary (p. 2454).
6. Supportive care: Provide suppo

## Automated Output Evaluation (LLM as the judge)

### Response Evaluation (Refined 1-5 Scoring with evidence)

**Process:** Each grounded answer is scored for groundedness and relevance using machine-parseable 1–5 judge outputs.

**Analysis:** In this saved run, groundedness averaged **4.8/5** with **7 of 9** questions scoring **5/5**. Relevance averaged **3.9/5**, with the weakest performance on **hypertension trade names (2/5)** and **patchy hair loss (3/5)**.

**Outcome:** The notebook now quantifies both evidence adherence and answer usefulness at the question level.

In [ ]:
# ============================================================
# Response Evaluation (Refined 1-5 Scoring with evidence)
# ============================================================

import re

groundedness_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context

Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluaton criteria and assign a score.
5. IMPORTANT: End your response with a single line: SCORE: X (where X is 1, 2, 3, 4, or 5)
"""

relevance_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context.
Consider whether all and only the important aspects are contained in the answer when evaluating relevance.

Instructions:
1. First write down the steps that are needed to evaluate the context as per the metric.
2. Give a step-by-step explanation if the context adheres to the metric considering the question as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the context using the evaluaton criteria and assign a score.
5. IMPORTANT: End your response with a single line: SCORE: X (where X is 1, 2, 3, 4, or 5)
"""

def evaluate_response(query, answer, context):
    # Groundedness Prompt
    ground_prompt = f"[INST] {groundedness_rater_system_message}\n\n###Question\n{query}\n###Context\n{context}\n###Answer\n{answer}\n[/INST]"

    # Use Main LLM (Mistral)
    ground_out = llm(ground_prompt, max_tokens=512, temperature=0.1)['choices'][0]['text'].strip()

    # Relevance Prompt
    rel_prompt = f"[INST] {relevance_rater_system_message}\n\n###Question\n{query}\n###Context\n{context}\n###Answer\n{answer}\n[/INST]"

    # Use Main LLM (Mistral)
    rel_out = llm(rel_prompt, max_tokens=512, temperature=0.1)['choices'][0]['text'].strip()

    # Parse Scores using Regex (Prioritize explicit format)
    def parse_score(text):
        # Try strict format first: SCORE: 5
        strict_matches = re.findall(r'SCORE:\s*([1-5])', text, re.IGNORECASE)
        if strict_matches:
            return int(strict_matches[-1])

        # Fallback: Look for last digit 1-5
        matches = re.findall(r'\b([1-5])\b', text)
        if matches:
            return int(matches[-1])
        return 0 # Error or not found

    ground_score = parse_score(ground_out)
    rel_score = parse_score(rel_out)

    return ground_score, ground_out, rel_score, rel_out

print("--- Evaluation Results ---\n")

eval_results_data = [] # We save the scores here so the UI cell can read them instantly

for i, (q, a, c) in enumerate(zip(questions, rag_responses, rag_contexts), 1):
    gs, ge, rs, re_text = evaluate_response(q, a, c)

    eval_results_data.append({
        "groundedness_score": gs,
        "relevance_score": rs
    })

    print(f"Q{i}: {q}")
    print(f"Groundedness Score: {gs}/5")
    print(f"Reason: {ge}")
    print(f"Relevance Score: {rs}/5")
    print(f"Reason: {re_text}")
    print("-" * 50)


--- Evaluation Results ---



Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Q1: What are the common symptoms and treatments for pulmonary embolism?
Groundedness Score: 5/5
Reason: Steps to evaluate the answer:
1. Identify the question and context provided.
2. Extract the common symptoms and treatments for pulmonary embolism from the context.
3. Compare the extracted information with the AI generated answer.
4. Determine if the answer is derived only from the context.
5. Rate the extent to which the metric is followed.
6. Assign a score based on the evaluation criteria.

The answer adheres to the metric as it is derived directly from the information presented in the context, specifically the sections: Introduction and Treatment. Therefore, the metric is followed completely.

SCORE: 5
Relevance Score: 5/5
Reason: Steps to evaluate the context as per the metric:
1. Identify the main aspects of the question: common symptoms and treatments for pulmonary embolism.
2. Read through the context to find information related to the question.
3. Determine if all important 

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Q2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?
Groundedness Score: 5/5
Reason: Steps to evaluate the answer:
1. Identify the common symptoms of appendicitis mentioned in the context.
2. Check if the AI generated answer includes those symptoms.
3. Determine if the AI generated answer mentions that appendicitis cannot be cured via medicine and requires surgical removal.
4. Verify if the surgical procedure for treating appendicitis, as stated in the answer, is consistent with what's mentioned in the context.
5. Ensure that all information in the answer is derived solely from the context.

The AI generated answer adheres to the metric as it includes all common symptoms of appendicitis mentioned in the context and correctly states that appendicitis cannot be cured via medicine and requires surgical removal, along with specifying an appendectomy as the typical procedure for treating appendic

...

Q7: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?
Groundedness Score: 4/5
Reason: Steps to evaluate the answer:
1. Identify the necessary precautions and treatment steps for a person who has fractured their leg based on the context.
2. Compare the identified steps with the AI generated answer to ensure that the answer is derived only from the information presented in the context.
3. Evaluate the extent to which the metric is followed by the answer.

Explanation:
The AI generated answer adheres to the metric as it provides a list of necessary precautions and treatment steps for a person who has fractured their leg based on the information presented in the context from The Merck Manual of Diagnosis & Therapy, 19th Edition. The answer does not introduce any new or irrelevant information.

Evaluation:
The metric is followed mostly as the AI generated answer inc

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Q8: What are the diagnostic steps for suspected endocrine disorders?
Groundedness Score: 5/5
Reason: Steps to evaluate the answer:
1. Identify the information related to diagnostic steps for suspected endocrine disorders in the context.
2. Compare the identified information with the AI generated answer.
3. Determine if the AI generated answer is derived only from the information presented in the context.

The AI generated answer adheres to the metric as it lists the diagnostic steps directly from the context. The answer mentions measuring levels of peripheral endocrine hormones and pituitary hormones, detecting autoantibodies, checking for elevated levels of pituitary tropic hormones, lifelong follow-up, and family screening - all of which are mentioned in the context.

Evaluation: The metric is followed completely.

Rating: 5 (The answer follows the metric completely)

SCORE: 5
Relevance Score: 5/5
Reason: Steps to evaluate the context as per the metric:
1. Identify the main aspects o

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Q9: What is the protocol for managing sepsis in a critical care unit?
Groundedness Score: 5/5
Reason: Steps to evaluate the answer:
1. Identify the key information related to the protocol for managing sepsis from the context.
2. Compare the identified information with the steps listed in the AI generated answer.
3. Determine if each step in the AI generated answer is derived directly from the context.

The answer adheres to the metric as each step in the answer is directly derived from the context provided. The answer includes all necessary steps for managing sepsis in a critical care unit, including first aid measures, fluid resuscitation, antibiotics, surgical intervention, and supportive care. Therefore, the answer follows the metric completely.

SCORE: 5
Relevance Score: 4/5
Reason: Steps to evaluate the context as per the metric:
1. Identify the main aspects of the question related to managing sepsis in a critical care unit.
2. Determine if the context provides sufficient informat

### Results Presentation UI

**Process:** Final answers, retrieved context, and evaluation scores are structured into a review dataset and rendered as an HTML card-based interface.

**Analysis:** The cell packages the notebook output into a frontend-style artifact and falls back to `N/A` scores if evaluation data is missing.

**Outcome:** Results are presented in a readable format for quick executive review and technical inspection.

In [ ]:
import pandas as pd
from IPython.display import display, HTML

# 1. Structure the data for frontend use (List of Dictionaries / DataFrame)
frontend_data = []

# Fallback in case eval_results_data wasn't generated yet
if 'eval_results_data' not in globals() or len(eval_results_data) != len(questions):
    eval_results_data = [{"groundedness_score": "N/A", "relevance_score": "N/A"} for _ in range(len(questions))]

for i, (q, a, c, e) in enumerate(zip(questions, rag_responses, rag_contexts, eval_results_data), 1):
    frontend_data.append({
        "id": i,
        "question": q,
        "answer": a,
        "context": c,
        "groundedness_score": e["groundedness_score"],
        "relevance_score": e["relevance_score"]
    })

# Convert to DataFrame for structured tabular use or JSON export later
results_df = pd.DataFrame(frontend_data)

# 2. Present in a readable, good-looking HTML UI
html_str = """
<style>
    .qa-card { border: 1px solid #e0e0e0; border-radius: 8px; margin-bottom: 16px; padding: 16px; font-family: sans-serif; box-shadow: 0 2px 4px rgba(0,0,0,0.05); }
    .qa-question { font-size: 1.1em; font-weight: bold; color: #1a73e8; margin-bottom: 12px; }
    .qa-answer { background-color: #f1f3f4; padding: 12px; border-left: 4px solid #1a73e8; border-radius: 0 4px 4px 0; margin-bottom: 12px; color: #202124; line-height: 1.5; }
    .qa-scores { margin-bottom: 12px; font-size: 0.95em; }
    .score-badge { display: inline-block; padding: 4px 8px; border-radius: 12px; font-weight: bold; margin-right: 8px; }
    .score-high { background-color: #ceead6; color: #0d652d; }
    .score-med { background-color: #fef7e0; color: #b06000; }
    .qa-context-summary { cursor: pointer; color: #188038; font-weight: 500; user-select: none; }
    .qa-context { background-color: #e6f4ea; padding: 12px; border-radius: 4px; font-size: 0.9em; margin-top: 8px; color: #137333; white-space: pre-wrap; line-height: 1.4; }
</style>
<div style="max-width: 950px; margin: 0 auto;">
    <h2 style="font-family: sans-serif; color: #202124;">RAG Diagnostic Results & Evaluation</h2>
    <p style="font-family: sans-serif; color: #5f6368;">
        Structured output ready for API/Frontend consumption. Includes automated LLM-as-a-judge scores.
    </p>
"""

def get_score_class(score):
    try:
        if float(score) >= 4:
            return "score-high"
        return "score-med"
    except:
        return "score-med"

for item in frontend_data:
    html_str += f"""
    <div class="qa-card">
        <div class="qa-question">Q{item['id']}: {item['question']}</div>
        <div class="qa-scores">
            <span class="score-badge {get_score_class(item['groundedness_score'])}">Groundedness: {item['groundedness_score']}/5</span>
            <span class="score-badge {get_score_class(item['relevance_score'])}">Relevance: {item['relevance_score']}/5</span>
        </div>
        <div class="qa-answer"><strong>Generated Answer:</strong><br/><br/>{item['answer']}</div>
        <details>
            <summary class="qa-context-summary">View Retrieved Context</summary>
            <div class="qa-context">{item['context']}</div>
        </details>
    </div>
    """

html_str += "</div>"

# Render the UI
display(HTML(html_str))


## Expanded Executive Summary

### TLDR

The Medical Diagnostic Assistant shows that grounding a local LLM in **The Merck Manual** materially improves answer discipline and traceability. In the current run, the system processed **4,114 pages**, indexed **18,020 chunks**, and the final RAG configuration (**`k=3`, `temp=0.1`**) achieved average **groundedness = 4.8/5** and **relevance = 3.9/5** across 9 clinical queries.

### Full Summary

**Objective:** This project addresses a core limitation of medical generative AI: fluent answers that are not reliably tied to authoritative evidence. The goal was to build a medical question-answering workflow that retrieves source passages from **The Merck Manual** and constrains answer generation to that evidence.

**Iterative Development:** The notebook progresses from an ungrounded Mistral baseline, to prompt tuning, to a retrieval pipeline built over **4,114 pages** of medical reference content and indexed into **18,020 chunks** for semantic search in **Chroma**. Final answer generation uses **`k=3`** and **`temperature=0.1`**.

**Performance Analysis:** The grounded system performed strongly on answer discipline and traceability. In this run, average **groundedness was 4.8/5** and average **relevance was 3.9/5**. Groundedness was the standout result, with **7 of 9** questions scoring a perfect **5/5**, showing that the system stayed closely anchored to retrieved evidence. The weakest relevance results appeared on **hypertension trade names (2/5)** and **patchy hair loss (3/5)**, indicating remaining retrieval-coverage limitations rather than broad hallucination failure.

**Economic Impact:** The practical value is faster access to trusted medical information and more defensible AI outputs. That can reduce search burden, improve consistency, and lower operational risk compared with unsupported medical generation.

**Deployment Readiness:** The architecture is best positioned as a **clinical decision-support layer**, not a substitute for clinician judgment. It already produces answers, retrieved evidence, and structured review outputs, making it a solid base for internal tools or clinician-facing reference products. The same integration paths can also support underdeveloped countries by helping clinicians focus limited resources more effectively on top of existing care infrastructure.

**Next Steps:** Improve weaker question coverage by broadening the medical corpus, strengthening retrieval for nomenclature-heavy and broad clinical prompts, adding human review to complement automated scoring, and productizing the output into a searchable interface with audit logging.